**`ingest_admin`**

Script examples to import administrative subdivisions for new countries

In [ ]:
from openplaces.api import get_admin1, get_admin2, get_admin3
from openplaces.core.schema import AdminId
from openplaces.io.ingest import ingest_recipe_data
from openplaces.recipe import get_recipe
from openplaces.timing import get_timer
from openplaces.utils import pretty_print

In [ ]:
REDO = False

In [ ]:
timer = get_timer('ingest_admin', verbose=True)

# Ingest data

In [ ]:
admin_id = AdminId('US')

## ``admin1``: states / departments

In [ ]:
admin1_recipe = get_recipe(admin_id, 'admin-nhgis-2020', filename='admin1')
pretty_print(admin1_recipe)

In [ ]:
ingest_recipe_data(admin1_recipe, timer=timer, redo=True)

In [ ]:
admin1 = get_admin1(recipe=admin1_recipe)
admin1.head()

## ``admin2``: counties / municipalities

In [ ]:
admin2_recipe = get_recipe(admin_id, 'admin-nhgis-2020', filename='admin2')
pretty_print(admin2_recipe)

In [ ]:
ingest_recipe_data(admin2_recipe, timer=timer, redo=True)

In [ ]:
admin2 = get_admin2(recipe=admin2_recipe, geom=True)

In [ ]:
admin2.sample(5).sort_index()

## ``admin3``: towns / county subdivisions

In [ ]:
admin3_recipe = get_recipe(admin_id, 'admin-nhgis-2020', filename='admin3')
admin3_recipe

In [ ]:
ingest_recipe_data(admin3_recipe, timer=timer, redo=True)

In [ ]:
admin3_local = get_admin3(recipe=admin3_recipe, all_columns=True)

admin3_local.head()

In [ ]:
admin3_local = get_admin3(recipe=admin3_recipe, all_columns=True)

# Create county FIPS (admin2_id_admin0)
admin3_local['admin2_id_admin0'] = admin3_local['admin3_id_admin0'].str.slice(0, 5)

# Join admin2_ids on county FIPS (admin2_id_admin0)
admin2_recipe = get_recipe('US', 'admin-nhgis-2020', filename='admin2')
admin2_crosswalk = (
    get_admin2(recipe=admin2_recipe, columns=['admin2_id_admin0'])
    .reset_index()
    .set_index('admin2_id_admin0')['admin2_id']
)
admin3_local = admin3_local.join(admin2_crosswalk, on='admin2_id_admin0')
admin3_local

# Test

In [ ]:
from openplaces.io.admin import generate_admin_ids

result = generate_admin_ids(admin3_local, 'admin3_id', 'admin2_id')
result
timer.mark('Done')
result[['name', 'name_long']].sample(25)